<a href="https://colab.research.google.com/github/hramirezq/melanoma-detection/blob/main/notebooks/train_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Entrenamiento en Colab (GPU T4)

Corre `src/train.py` para las corridas del Stage 2. Todo se guarda directo en Drive (`MyDrive/melanoma-detection/results/`), así que si Colab corta la sesión basta con volver a ejecutar: las corridas terminadas se saltan y la que estaba a medias sigue con `--resume`.

**Antes de empezar**
1. *Entorno de ejecución → Cambiar tipo → GPU T4*.
2. Subir `data/raw/pad_256.zip` (generado con `python -m src.data --resize data/raw/pad_256` y comprimido) a `MyDrive/melanoma-detection/data/pad_256.zip`.

**Valores esperados (T4):** humo de 1 época en ~2 min; cada corrida `effb0` en ~5–10 min (30 épocas máx. con early stopping) y `vits16` en ~10–15 min. La mejor época no debería ser la última y la AUC cáncer en `val` debería quedar en torno a 0,85–0,92.

In [ ]:
import os, json, glob
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/melanoma-detection'
OUT = f'{DRIVE}/results'

if not os.path.exists('/content/melanoma-detection'):
    !git clone -q https://github.com/hramirezq/melanoma-detection.git /content/melanoma-detection
%cd /content/melanoma-detection
!git pull -q
!pip install -q timm==1.0.30
if not os.path.exists('data/raw/pad_256'):
    !unzip -q {DRIVE}/data/pad_256.zip -d data/raw/
!ls data/raw/pad_256 | wc -l
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Prueba de humo (M1)
Una época; verifica el contrato de predicciones. Se escribe en `/content/smoke`, fuera de Drive.

In [ ]:
!python -m src.train configs/effb0_standard_s1.yaml --epochs 1 --out /content/smoke --workers 2

p = pd.read_csv('/content/smoke/preds/effb0_standard_s1.csv')
cols = ['img_id', 'split', 'corruption', 'severity', 'y_true'] + [f'p_{c}' for c in ['ACK', 'BCC', 'MEL', 'NEV', 'SCC', 'SEK']]
assert list(p.columns) == cols, p.columns
assert len(p) == 329 + 329 * 31, len(p)
assert (p.filter(like='p_').sum(axis=1) - 1).abs().max() < 1e-3
print('humo OK:', json.load(open('/content/smoke/logs/effb0_standard_s1.json'))['gpu'])

## 2. Corridas
A (CNN): `MODELS = ['effb0']`. B (ViT): `MODELS = ['vits16']`. Si falta tiempo, recortar `SEEDS` a `[1, 2]` antes que quitar una *augmentation*.

In [ ]:
MODELS = ['effb0']
AUGS = ['none', 'standard', 'robust']
SEEDS = [1, 2, 3]

for m in MODELS:
    for a in AUGS:
        for s in SEEDS:
            run = f'{m}_{a}_s{s}'
            if os.path.exists(f'{OUT}/preds/{run}.csv'):
                print('ya existe', run)
                continue
            !python -m src.train configs/{run}.yaml --out {OUT} --resume --workers 2

## 3. Resumen
Copiar la salida de esta celda (o devolver el notebook ejecutado). Las métricas de `test` las calcula `src/evaluate.py` en local; aquí solo se mira `val`.

In [ ]:
rows = []
for f in sorted(glob.glob(f'{OUT}/logs/*.json')):
    l = json.load(open(f))
    n = len(pd.read_csv(f"{OUT}/preds/{l['run_id']}.csv"))
    rows.append(dict(run=l['run_id'], best_epoch=l['best_epoch'], epochs=l['epochs_run'],
                     val_auc_cancer=round(l['best_val_auc_cancer'], 4),
                     val_bacc=round(l['history'][l['best_epoch'] - 1]['val_bacc'], 4),
                     minutes=round(l['wall_seconds'] / 60, 1), pred_rows=n, gpu=l['gpu']))
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
print('total GPU h:', round(summary['minutes'].sum() / 60, 2))

## 4. Descargar predicciones y logs
Descomprimir en la raíz del repo local (`results/preds`, `results/logs`) y commitear desde ahí. Los checkpoints se quedan en Drive.

In [ ]:
tag = '_'.join(MODELS)
!cd {OUT} && zip -qr /content/results_{tag}.zip preds logs
from google.colab import files
files.download(f'/content/results_{tag}.zip')

## 5. HAM10000 → PAD (H3)
Entrena EfficientNet-B0 (`standard`) en HAM10000 con early stopping en su propia validación (por lesión) y predice `val` y `test` de PAD con las mismas corrupciones. Requiere en Drive `data/ham10000/` con `HAM10000_images_part_1.zip`, `HAM10000_images_part_2.zip` y `HAM10000_metadata.tab`.

**Valores esperados (T4):** 10015 imágenes encontradas; ~1 min por época, ~15–30 min en total. La AUC cáncer en la validación de HAM debería quedar alta (≳ 0,90); la de PAD se mide después en local y se espera claramente menor.

In [ ]:
from google.colab import files
HAM = '/content/ham10000'
if not os.path.exists(HAM):
    !mkdir -p {HAM} && cp {DRIVE}/data/ham10000/HAM10000_metadata.* {HAM}/
    !unzip -q {DRIVE}/data/ham10000/HAM10000_images_part_1.zip -d {HAM}
    !unzip -q {DRIVE}/data/ham10000/HAM10000_images_part_2.zip -d {HAM}
!find {HAM} -name '*.jpg' | wc -l

run = 'ham_effb0_standard_s1'
if not os.path.exists(f'{OUT}/preds/{run}.csv'):
    !python -m src.train configs/{run}.yaml --ham-root {HAM} --out {OUT} --resume --workers 2

l = json.load(open(f'{OUT}/logs/{run}.json'))
print(run, 'best epoch', l['best_epoch'], 'HAM val AUC cancer', round(l['best_val_auc_cancer'], 4),
      'min', round(l['wall_seconds'] / 60, 1), 'pred_rows', len(pd.read_csv(f'{OUT}/preds/{run}.csv')))
!cd {OUT} && zip -q /content/results_ham.zip preds/{run}.csv logs/{run}.json
files.download('/content/results_ham.zip')